# Seed a source

Loads people or policies into the HR harness through `POST /admin/seed`. The service generates the rows with Faker and writes them with SQLAlchemy, so this notebook never opens the SQLite file.

HR is built. Sales, Product, and Engineering do not expose this endpoint yet.

## Parameters

In [1]:
import os

SOURCE = os.environ.get("SEED_SOURCE", "hr")
KIND = os.environ.get("SEED_KIND", "person")  # person or policy
COUNT = int(os.environ.get("SEED_COUNT", "10000"))
SEED = int(os.environ.get("SEED_VALUE", "1"))
RESET = os.environ.get("SEED_RESET", "false").lower() == "true"
print(SOURCE, KIND, COUNT, SEED, RESET)


hr person 10000 1 False


## Load

In [2]:
import httpx

URLS = {"hr": os.environ.get("HARNESS_HR_URL", "http://127.0.0.1:8101")}
if SOURCE not in URLS:
    raise SystemExit(f"{SOURCE} has no seed endpoint yet")
base = URLS[SOURCE]
if RESET:
    httpx.post(f"{base}/admin/reset", timeout=120).raise_for_status()
response = httpx.post(
    f"{base}/admin/seed",
    json={"kind": KIND, "count": COUNT, "seed": SEED, "replace": False},
    timeout=600,
)
response.raise_for_status()
result = response.json()
elapsed = result['elapsed_seconds'] or 0
rate = result['inserted'] / elapsed if elapsed else 0
print(result)
print(f"{result['inserted']} {KIND} rows in {elapsed}s ({rate:.0f} rows/s)")


{'kind': 'person', 'inserted': 10000, 'seed': 1, 'counts': {'person': 10048, 'policy': 11}, 'elapsed_seconds': 2.2}
10000 person rows in 2.2s (4545 rows/s)


## Check the load

In [3]:
discover = httpx.get(f"{base}/discover", timeout=60).json()
ids = httpx.get(f"{base}/ids", timeout=120).json()
print("counts", discover["counts"])
print("ids", {kind: len(rows) for kind, rows in ids["ids"].items()})
assert discover["counts"][KIND] == len(ids["ids"][KIND])

seen = set()
token = None
while True:
    params = {"limit": 500}
    if token:
        params["page_token"] = token
    page = httpx.get(f"{base}/changes", params=params, timeout=60).json()
    for item in page["items"]:
        if item["object_type"] == KIND:
            seen.add(item["id"])
    token = page["next_page_token"]
    if not token:
        break
print("paged", KIND, len(seen))
assert len(seen) == discover["counts"][KIND]


counts {'person': 10048, 'policy': 11}
ids {'person': 10048, 'policy': 11}
paged person 10048
